# Acquire and take sequence

This notebook runs the LatissAcquireAndTakeSequence script from within a notebook

Craig Lage - 26Mar21

In [1]:
import sys
import asyncio
import time
import astropy
import numpy as np
import logging 
import yaml

from lsst.ts import salobj
from lsst.ts.idl.enums.Script import ScriptState
from lsst.ts.externalscripts.auxtel.latiss_acquire_and_take_sequence import LatissAcquireAndTakeSequence
from lsst.ts.observatory.control.utils import RotType

In [2]:
stream_handler = logging.StreamHandler(sys.stdout)
# if you want logging
logger = logging.getLogger()
logger.addHandler(stream_handler)
logger.level = logging.DEBUG

# turn off logging for matplotlib
mpl_logger = logging.getLogger('matplotlib')
mpl_logger.setLevel(logging.WARNING)

In [ ]:
# Note that you need to restart here if the script completes

In [3]:
script = LatissAcquireAndTakeSequence(index=2)  # this essentially calls the init method
# script = LatissAcquireAndTakeSequence(index=os.getuid())

DEBUG:Script.ATCS:atmcs: Adding all resources.
atmcs: Adding all resources.
DEBUG:Script.ATCS:atptg: Adding all resources.
atptg: Adding all resources.
DEBUG:Script.ATCS:ataos: Adding all resources.
ataos: Adding all resources.
DEBUG:Script.ATCS:atpneumatics: Adding all resources.
atpneumatics: Adding all resources.
DEBUG:Script.ATCS:athexapod: Adding all resources.
athexapod: Adding all resources.
DEBUG:Script.ATCS:atdome: Adding all resources.
atdome: Adding all resources.
DEBUG:Script.ATCS:atdometrajectory: Adding all resources.
atdometrajectory: Adding all resources.
DEBUG:Script.LATISS:atcamera: Adding all resources.
atcamera: Adding all resources.
DEBUG:Script.LATISS:atspectrograph: Adding all resources.
atspectrograph: Adding all resources.
DEBUG:Script.LATISS:atheaderservice: Adding all resources.
atheaderservice: Adding all resources.
DEBUG:Script.LATISS:atarchiver: Adding all resources.
atarchiver: Adding all resources.


In [4]:
# make sure all remotes etc are running
await script.start_task

INFO:ATPtg:Read historical data in 0.02 sec
Read historical data in 0.02 sec
DEBUG:ATPtg:Read 14 history items for RemoteEvent(ATPtg, 0, airmassWarning)
Read 14 history items for RemoteEvent(ATPtg, 0, airmassWarning)
DEBUG:ATPtg:Read 1 history items for RemoteEvent(ATPtg, 0, azWrapWarning)
Read 1 history items for RemoteEvent(ATPtg, 0, azWrapWarning)
DEBUG:ATPtg:Read 4 history items for RemoteEvent(ATPtg, 0, currentDebugLevel)
Read 4 history items for RemoteEvent(ATPtg, 0, currentDebugLevel)
DEBUG:ATPtg:Read 56 history items for RemoteEvent(ATPtg, 0, currentTarget)
Read 56 history items for RemoteEvent(ATPtg, 0, currentTarget)
DEBUG:ATPtg:Read 100 history items for RemoteEvent(ATPtg, 0, detailedState)
Read 100 history items for RemoteEvent(ATPtg, 0, detailedState)
DEBUG:ATPtg:Read 1 history items for RemoteEvent(ATPtg, 0, elLimitWarning)
Read 1 history items for RemoteEvent(ATPtg, 0, elLimitWarning)
DEBUG:ATPtg:Read 3 history items for RemoteEvent(ATPtg, 0, errorCode)
Read 3 history it

In [ ]:
# set wrap strategy
#script.atcs.rem.atptg.cmd_raDecTarget.set(azWrapStrategy=1)  # 1 does not unwrap, 0 unwraps

In [ ]:
# ATAOS must be on and corrections enabled, do as follows if required
# await script.atcs.rem.ataos.cmd_enableCorrection.set_start(m1=True, hexapod=True, atspectrograph=True)

# Emulate how the scriptQueue launches scripts

In [29]:
# Needed to run the script a 2nd time 
script.set_state(ScriptState.UNCONFIGURED)

In [30]:
configuration = yaml.safe_dump({"object_name": 'HR 5446',
                                "do_acquire": True,
                                "acq_filter" : 'RG610',
                                "acq_grating" : 'empty_1',
                                "exposure_time_sequence" : [4, 4, 2, 2], 
                                "filter_sequence": ['empty_1','quadnotch1', 'BG40','RG610'], 
                                "grating_sequence": ['ronchi90lpmm','ronchi90lpmm','empty_1','empty_1'],
                                "do_pointing_model": False,
                                "dataPath": '/project/shared/auxTel/rerun/quickLook',
                                "max_acq_iter": 4,
                                "target_pointing_verification": True,
                                "do_take_sequence": False})
print(configuration)

acq_filter: RG610
acq_grating: empty_1
dataPath: /project/shared/auxTel/rerun/quickLook
do_acquire: true
do_pointing_model: false
do_take_sequence: false
exposure_time_sequence:
- 4
- 4
- 2
- 2
filter_sequence:
- empty_1
- quadnotch1
- BG40
- RG610
grating_sequence:
- ronchi90lpmm
- ronchi90lpmm
- empty_1
- empty_1
max_acq_iter: 4
object_name: HR 5446
target_pointing_verification: true



In [31]:
config_data = script.cmd_configure.DataType()
config_data.config = configuration
await script.do_configure(config_data)

target DDS read queue is filling: 10 of 100 elements
target python read queue is filling: 32 of 100 elements


In [32]:
# Check summary of focus offsets
tmp=await script.atcs.rem.ataos.evt_focusOffsetSummary.aget()
print(tmp)

private_revCode: d8296953, private_sndStamp: 1621990391.5978308, private_rcvStamp: 1621990391.5984054, private_seqNum: 249, private_identity: ATAOS, private_origin: 153650, private_host: 0, total: -0.28055158257484436, userApplied: 0.00984840840101242, filter: -0.01600000075995922, disperser: -0.21639999747276306, wavelength: 0.0, priority: 0


In [33]:
# Check summary of pointing offsets
tmp=await script.atcs.rem.ataos.evt_pointingOffsetSummary.aget()
print(tmp)

private_revCode: e1bb3fb9, private_sndStamp: 1621990399.6686487, private_rcvStamp: 1621990399.669403, private_seqNum: 237, private_identity: ATAOS, private_origin: 153650, private_host: 0, total: [-16.5, 1.2999999523162842], filter: [0.0, 0.0], disperser: [-16.5, 1.2999999523162842], priority: 0


In [34]:
# Run the script
group_id_data = script.cmd_setGroupId.DataType(
                groupId=astropy.time.Time.now().isot
            )
await script.do_setGroupId(group_id_data)

run_data = script.cmd_run.DataType()
await script.arun()

DEBUG:Script:Beginning target acquisition
Beginning target acquisition
INFO:Script:Performing Blind offset set to True
Performing Blind offset set to True
DEBUG:Script:After slew, the target should be at the boresight [(2036.5, 2000.5)] whereas the target position is (1780, 1800). Blindly offsetting [26.8, 21.0] arcsec to this position.
After slew, the target should be at the boresight [(2036.5, 2000.5)] whereas the target position is (1780, 1800). Blindly offsetting [26.8, 21.0] arcsec to this position.
DEBUG:urllib3.connectionpool:Resetting dropped connection: simbad.u-strasbg.fr
Resetting dropped connection: simbad.u-strasbg.fr
DEBUG:urllib3.connectionpool:http://simbad.u-strasbg.fr:80 "POST /simbad/sim-script HTTP/1.1" 200 None
http://simbad.u-strasbg.fr:80 "POST /simbad/sim-script HTTP/1.1" 200 None
INFO:Script.ATCS:Slewing to HR 5446: 14 36 24.1038 -39 35 50.700
Slewing to HR 5446: 14 36 24.1038 -39 35 50.700
DEBUG:Script.ATCS:Setting rotator position with respect to parallactic 

In [35]:
await script.latiss.setup_atspec(filter='empty_1', grating='empty_1')

In [36]:
await script.atcs.rem.ataos.cmd_resetOffset.set_start(axis='z')

In [5]:
await script.latiss.setup_atspec(filter='RG610', grating='empty_1')

In [8]:
await script.latiss.take_object(5.0, 1, filter='RG610', grating='empty_1')

DEBUG:Script.LATISS:Generating group_id
Generating group_id
DEBUG:Script.LATISS:imagetype: OBJECT, wait for TCS to be ready.
imagetype: OBJECT, wait for TCS to be ready.
DEBUG:Script.ATCS:atspectrograph correction running. Trying to determine state of the corrections.
atspectrograph correction running. Trying to determine state of the corrections.
DEBUG:Script.ATCS:Received corrections: ['started', 'completed'].
Received corrections: ['started', 'completed'].
DEBUG:Script.ATCS:Ready to take data:: atmcs=True, ataos=False, atdome=True.
Ready to take data:: atmcs=True, ataos=False, atdome=True.
DEBUG:Script.ATCS:atspectrograph correction running. Trying to determine state of the corrections.
atspectrograph correction running. Trying to determine state of the corrections.
DEBUG:Script.ATCS:No correction seen in the last 5.0 seconds. Determining order of last corrections.
No correction seen in the last 5.0 seconds. Determining order of last corrections.
DEBUG:Script.ATCS:Last correction co

array([2021060800203])

In [20]:
await script.atcs.offset_xy(y=-150, x=0, relative=True)

DEBUG:Script.ATCS:Calculating x/y offset: 0/-150 
Calculating x/y offset: 0/-150 
DEBUG:Script.ATCS:Applying Az/El offset: -105.4657890855505/106.6628676370656 
Applying Az/El offset: -105.4657890855505/106.6628676370656 
DEBUG:Script.ATCS:Telescope not in position.
Telescope not in position.
DEBUG:Script.ATCS:All axes in position.
All axes in position.
DEBUG:Script.ATCS:Waiting for telescope to settle.
Waiting for telescope to settle.
DEBUG:Script.ATCS:Done
Done


In [35]:
await script.latiss.setup_atspec(filter='RG610', grating='empty_1')

In [36]:
await script.latiss.take_object(5.0, 1, filter='RG610', grating='empty_1')

DEBUG:Script.LATISS:Generating group_id
Generating group_id
DEBUG:Script.LATISS:imagetype: OBJECT, wait for TCS to be ready.
imagetype: OBJECT, wait for TCS to be ready.
DEBUG:Script.ATCS:atspectrograph correction running. Trying to determine state of the corrections.
atspectrograph correction running. Trying to determine state of the corrections.
DEBUG:Script.ATCS:Received corrections: ['started'].
Received corrections: ['started'].
DEBUG:Script.ATCS:Ready to take data:: atmcs=True, ataos=False, atdome=True.
Ready to take data:: atmcs=True, ataos=False, atdome=True.
DEBUG:Script.ATCS:atspectrograph correction running. Trying to determine state of the corrections.
atspectrograph correction running. Trying to determine state of the corrections.
DEBUG:Script.ATCS:Received corrections: ['started', 'completed'].
Received corrections: ['started', 'completed'].
DEBUG:Script.ATCS:Ready to take data:: atmcs=True, ataos=False, atdome=True.
Ready to take data:: atmcs=True, ataos=False, atdome=Tr

array([2021060800226])

In [25]:
await script.latiss.setup_atspec(filter='RG610', grating='ronchi90lpmm')

In [26]:
await script.latiss.take_object(5.0, 1, filter='RG610', grating='ronchi90lpmm')

DEBUG:Script.LATISS:Generating group_id
Generating group_id
DEBUG:Script.LATISS:imagetype: OBJECT, wait for TCS to be ready.
imagetype: OBJECT, wait for TCS to be ready.
DEBUG:Script.ATCS:atspectrograph correction running. Trying to determine state of the corrections.
atspectrograph correction running. Trying to determine state of the corrections.
DEBUG:Script.ATCS:Received corrections: ['started', 'completed'].
Received corrections: ['started', 'completed'].
DEBUG:Script.ATCS:Ready to take data:: atmcs=True, ataos=False, atdome=True.
Ready to take data:: atmcs=True, ataos=False, atdome=True.
DEBUG:Script.ATCS:atspectrograph correction running. Trying to determine state of the corrections.
atspectrograph correction running. Trying to determine state of the corrections.
DEBUG:Script.ATCS:No correction seen in the last 5.0 seconds. Determining order of last corrections.
No correction seen in the last 5.0 seconds. Determining order of last corrections.
DEBUG:Script.ATCS:Last correction co

array([2021060800221])

In [27]:
await script.latiss.setup_atspec(filter='RG610', grating='ronchi170lpmm')

In [28]:
await script.latiss.take_object(10.0, 1, filter='RG610', grating='ronchi170lpmm')

DEBUG:Script.LATISS:Generating group_id
Generating group_id
DEBUG:Script.LATISS:imagetype: OBJECT, wait for TCS to be ready.
imagetype: OBJECT, wait for TCS to be ready.
DEBUG:Script.ATCS:atspectrograph correction running. Trying to determine state of the corrections.
atspectrograph correction running. Trying to determine state of the corrections.
DEBUG:Script.ATCS:Received corrections: ['started', 'completed'].
Received corrections: ['started', 'completed'].
DEBUG:Script.ATCS:Ready to take data:: atmcs=True, ataos=False, atdome=True.
Ready to take data:: atmcs=True, ataos=False, atdome=True.
DEBUG:Script.ATCS:atspectrograph correction running. Trying to determine state of the corrections.
atspectrograph correction running. Trying to determine state of the corrections.
DEBUG:Script.ATCS:No correction seen in the last 5.0 seconds. Determining order of last corrections.
No correction seen in the last 5.0 seconds. Determining order of last corrections.
DEBUG:Script.ATCS:Last correction co

array([2021060800222])

In [39]:
await script.latiss.setup_atspec(filter='RG610', grating='holo4_003')

In [40]:
await script.latiss.take_object(10.0, 1, filter='RG610', grating='holo4_003')

DEBUG:Script.LATISS:Generating group_id
Generating group_id
DEBUG:Script.LATISS:imagetype: OBJECT, wait for TCS to be ready.
imagetype: OBJECT, wait for TCS to be ready.
DEBUG:Script.ATCS:atspectrograph correction running. Trying to determine state of the corrections.
atspectrograph correction running. Trying to determine state of the corrections.
DEBUG:Script.ATCS:Received corrections: ['started', 'completed'].
Received corrections: ['started', 'completed'].
DEBUG:Script.ATCS:Ready to take data:: atmcs=True, ataos=False, atdome=True.
Ready to take data:: atmcs=True, ataos=False, atdome=True.
DEBUG:Script.ATCS:atspectrograph correction running. Trying to determine state of the corrections.
atspectrograph correction running. Trying to determine state of the corrections.
DEBUG:Script.ATCS:No correction seen in the last 5.0 seconds. Determining order of last corrections.
No correction seen in the last 5.0 seconds. Determining order of last corrections.
DEBUG:Script.ATCS:Last correction co

array([2021060800233])

logMessage DDS read queue is filling: 11 of 100 elements
logMessage DDS read queue is filling: 55 of 100 elements
logMessage DDS read queue is filling: 16 of 100 elements
logMessage DDS read queue is filling: 81 of 100 elements
logMessage DDS read queue is filling: 79 of 100 elements
logMessage DDS read queue is filling: 10 of 100 elements
logMessage DDS read queue is filling: 10 of 100 elements
logMessage DDS read queue is filling: 13 of 100 elements
logMessage DDS read queue is filling: 11 of 100 elements
logMessage DDS read queue is filling: 10 of 100 elements
logMessage DDS read queue is filling: 11 of 100 elements
logMessage DDS read queue is filling: 14 of 100 elements
logMessage DDS read queue is filling: 12 of 100 elements
logMessage DDS read queue is filling: 13 of 100 elements
logMessage DDS read queue is filling: 10 of 100 elements
logMessage DDS read queue is filling: 14 of 100 elements
logMessage DDS read queue is filling: 11 of 100 elements
logMessage DDS read queue is fi

In [59]:
await script.atcs.rem.ataos.cmd_disableCorrection.set_start(m1=True, hexapod=True)

In [68]:
await salobj.set_summary_state(script.latiss.rem.atspectrograph, salobj.State.ENABLED, settingsToApply='current')

[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]

In [62]:
await script.atcs.rem.ataos.cmd_enableCorrection.set_start(m1=True, hexapod=True, atspectrograph=True)

In [75]:
await script.latiss.take_object(10.0, 1, filter='RG610', grating='holo4_003')

DEBUG:Script.LATISS:Generating group_id
Generating group_id
DEBUG:Script.LATISS:imagetype: OBJECT, wait for TCS to be ready.
imagetype: OBJECT, wait for TCS to be ready.
DEBUG:Script.ATCS:atspectrograph correction running. Trying to determine state of the corrections.
atspectrograph correction running. Trying to determine state of the corrections.
DEBUG:Script.ATCS:Received corrections: ['started'].
Received corrections: ['started'].
DEBUG:Script.ATCS:Ready to take data:: atmcs=False, ataos=False, atdome=True.
Ready to take data:: atmcs=False, ataos=False, atdome=True.
DEBUG:Script.ATCS:atspectrograph correction running. Trying to determine state of the corrections.
atspectrograph correction running. Trying to determine state of the corrections.
DEBUG:Script.ATCS:Received corrections: ['completed'].
Received corrections: ['completed'].
DEBUG:Script.ATCS:Ready to take data:: atmcs=True, ataos=False, atdome=True.
Ready to take data:: atmcs=True, ataos=False, atdome=True.
DEBUG:Script.ATC

array([2021052500250])

In [71]:
await script.atcs.rem.ataos.cmd_offset.set_start(z=0.22)

In [47]:
await script.latiss.take_object(2.0, 1, filter='RG610', grating='ronchi170lpmm')

DEBUG:Script.LATISS:Generating group_id
Generating group_id
DEBUG:Script.LATISS:imagetype: OBJECT, wait for TCS to be ready.
imagetype: OBJECT, wait for TCS to be ready.
DEBUG:Script.ATCS:atspectrograph correction running. Trying to determine state of the corrections.
atspectrograph correction running. Trying to determine state of the corrections.
DEBUG:Script.ATCS:Received corrections: ['started'].
Received corrections: ['started'].
DEBUG:Script.ATCS:Ready to take data:: atmcs=False, ataos=False, atdome=True.
Ready to take data:: atmcs=False, ataos=False, atdome=True.
DEBUG:Script.ATCS:atspectrograph correction running. Trying to determine state of the corrections.
atspectrograph correction running. Trying to determine state of the corrections.
DEBUG:Script.ATCS:Received corrections: ['completed'].
Received corrections: ['completed'].
DEBUG:Script.ATCS:Ready to take data:: atmcs=True, ataos=False, atdome=True.
Ready to take data:: atmcs=True, ataos=False, atdome=True.
DEBUG:Script.ATC

array([2021052500205])

In [42]:
await script.latiss.take_object(2.0, 1, filter='RG610', grating='holo4_003')

DEBUG:Script.LATISS:Generating group_id
Generating group_id
DEBUG:Script.LATISS:imagetype: OBJECT, wait for TCS to be ready.
imagetype: OBJECT, wait for TCS to be ready.
DEBUG:Script.ATCS:atspectrograph correction running. Trying to determine state of the corrections.
atspectrograph correction running. Trying to determine state of the corrections.
DEBUG:Script.ATCS:Received corrections: ['started'].
Received corrections: ['started'].
DEBUG:Script.ATCS:Ready to take data:: atmcs=False, ataos=False, atdome=True.
Ready to take data:: atmcs=False, ataos=False, atdome=True.
DEBUG:Script.ATCS:atspectrograph correction running. Trying to determine state of the corrections.
atspectrograph correction running. Trying to determine state of the corrections.
DEBUG:Script.ATCS:Received corrections: ['completed'].
Received corrections: ['completed'].
DEBUG:Script.ATCS:Ready to take data:: atmcs=True, ataos=False, atdome=True.
Ready to take data:: atmcs=True, ataos=False, atdome=True.
DEBUG:Script.ATC

array([2021052500202])

In [44]:
await script.latiss.setup_atspec(filter='RG610', grating='ronchi170lpmm')

In [45]:
await script.latiss.take_object(2.0, 1, filter='RG610', grating='ronchi170lpmm')

DEBUG:Script.LATISS:Generating group_id
Generating group_id
DEBUG:Script.LATISS:imagetype: OBJECT, wait for TCS to be ready.
imagetype: OBJECT, wait for TCS to be ready.
DEBUG:Script.ATCS:atspectrograph correction running. Trying to determine state of the corrections.
atspectrograph correction running. Trying to determine state of the corrections.
DEBUG:Script.ATCS:Received corrections: ['started', 'completed'].
Received corrections: ['started', 'completed'].
DEBUG:Script.ATCS:Ready to take data:: atmcs=True, ataos=False, atdome=True.
Ready to take data:: atmcs=True, ataos=False, atdome=True.
DEBUG:Script.ATCS:atspectrograph correction running. Trying to determine state of the corrections.
atspectrograph correction running. Trying to determine state of the corrections.
DEBUG:Script.ATCS:No correction seen in the last 5.0 seconds. Determining order of last corrections.
No correction seen in the last 5.0 seconds. Determining order of last corrections.
DEBUG:Script.ATCS:Last correction co

array([2021052500203])

In [ ]:
await script.atcs.stop_tracking()